In [1]:
import pandas as pd
import numpy as np
import joblib

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(
    "../data/processed/fraud_processed.csv"
)

print(df.shape)

df.head()

(6362620, 24)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,balance_error_org,balance_error_dest,customer_type,merchant_type,age_group,customer_segment,channel,device_type,risk_region,account_tenure_months
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,...,1.455192e-11,-9839.64,C,M,26-35,Retail,Mobile App,iPhone,Central,20
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,...,-1.136868e-12,-1864.28,C,M,60+,Retail,Internet Banking,Android,East,7
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,...,0.000000e+00,-181.00,C,C,36-45,Retail,Internet Banking,Desktop,East,39
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,...,0.000000e+00,-21363.00,C,C,36-45,Premium,Mobile App,Android,North,116
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,...,0.000000e+00,-11668.14,C,M,18-25,Retail,Internet Banking,iPhone,East,19


In [3]:
model = joblib.load(
    "../models/random_forest_fraud_detector.pkl"
)

print("Model Loaded Successfully")

Model Loaded Successfully


In [4]:
df_model = df.copy()

df_model = df_model.drop(
    columns=[
        "nameOrig",
        "nameDest"
    ]
)

In [5]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "type",
    "customer_type",
    "merchant_type",
    "age_group",
    "customer_segment",
    "channel",
    "device_type",
    "risk_region"
]

for col in categorical_cols:

    le = LabelEncoder()

    df_model[col] = le.fit_transform(
        df_model[col].astype(str)
    )
    
X = df_model.drop(
    columns=["isFraud"]
)

print(X.shape)

(6362620, 21)


In [6]:
fraud_probability = model.predict_proba(
    X
)[:,1]

df["fraud_probability"] = fraud_probability

df[
    ["fraud_probability"]
].head()

,fraud_probability
0,0.000000
1,0.000000
2,0.729989
3,0.589770
4,0.000000


In [7]:
df["risk_score"] = (
    df["fraud_probability"] * 100
).round(2)

In [8]:
def risk_category(score):

    if score >= 90:
        return "Critical"

    elif score >= 70:
        return "High"

    elif score >= 40:
        return "Medium"

    else:
        return "Low"



In [9]:
df["risk_category"] = df[
    "risk_score"
].apply(
    risk_category
)

In [10]:
def alert_priority(category):

    if category == "Critical":
        return "Immediate"

    elif category == "High":
        return "Urgent"

    elif category == "Medium":
        return "Review"

    else:
        return "Normal"

In [11]:
df["alert_priority"] = df[
    "risk_category"
].apply(
    alert_priority
)

In [12]:
def recommendation(row):

    score = row["risk_score"]

    if score >= 90:
        return (
            "Block Transaction Immediately "
            "and Escalate Investigation"
        )

    elif score >= 70:
        return (
            "Require Multi-Factor Verification"
        )

    elif score >= 40:
        return (
            "Manual Review Recommended"
        )

    else:
        return (
            "Approve Transaction"
        )

In [13]:
df["recommended_action"] = df.apply(
    recommendation,
    axis=1
)

In [14]:
def alert_message(row):

    score = row["risk_score"]

    if score >= 90:

        return (
            f"CRITICAL FRAUD ALERT | "
            f"Risk Score: {score}"
        )

    elif score >= 70:

        return (
            f"HIGH RISK TRANSACTION | "
            f"Risk Score: {score}"
        )

    elif score >= 40:

        return (
            f"MEDIUM RISK TRANSACTION | "
            f"Risk Score: {score}"
        )

    else:

        return (
            f"LOW RISK TRANSACTION | "
            f"Risk Score: {score}"
        )

In [15]:
df["alert_message"] = df.apply(
    alert_message,
    axis=1
)

In [16]:
df["review_required"] = np.where(
    df["risk_score"] >= 40,
    "Yes",
    "No"
)

df["investigation_required"] = np.where(
    df["risk_score"] >= 70,
    "Yes",
    "No"
)

In [17]:
df[
    [
        "fraud_probability",
        "risk_score",
        "risk_category",
        "alert_priority",
        "recommended_action",
        "review_required",
        "investigation_required"
    ]
].head(10)

,fraud_probability,risk_score,risk_category,alert_priority,recommended_action,review_required,investigation_required
0,0.000000,0.00,Low,Normal,Approve Transaction,No,No
1,0.000000,0.00,Low,Normal,Approve Transaction,No,No
2,0.729989,73.00,High,Urgent,Require Multi-Factor Verification,Yes,Yes
3,0.589770,58.98,Medium,Review,Manual Review Recommended,Yes,No
4,0.000000,0.00,Low,Normal,Approve Transaction,No,No
5,0.000000,0.00,Low,Normal,Approve Transaction,No,No
6,0.000000,0.00,Low,Normal,Approve Transaction,No,No
7,0.000000,0.00,Low,Normal,Approve Transaction,No,No
8,0.040000,4.00,Low,Normal,Approve Transaction,No,No
9,0.009921,0.99,Low,Normal,Approve Transaction,No,No


In [18]:
df["risk_category"].value_counts()

risk_category
Low         6354427
Critical       7352
High            732
Medium          109
Name: count, dtype: int64

In [19]:
df[
    [
        "risk_score",
        "risk_category",
        "alert_priority"
    ]
].head()

,risk_score,risk_category,alert_priority
0,0.00,Low,Normal
1,0.00,Low,Normal
2,73.00,High,Urgent
3,58.98,Medium,Review
4,0.00,Low,Normal


In [20]:
def fraud_severity(score):

    if score >= 95:
        return "Extreme"

    elif score >= 80:
        return "Severe"

    elif score >= 60:
        return "Moderate"

    elif score >= 40:
        return "Minor"

    else:
        return "None"

df["fraud_severity"] = df["risk_score"].apply(
    fraud_severity
)

df["fraud_severity"].value_counts()

fraud_severity
None        6354427
Extreme        6630
Severe         1248
Moderate        282
Minor            33
Name: count, dtype: int64

In [21]:
df.to_csv(
    "../data/processed/fraud_risk_scored.csv",
    index=False
)

print("Risk Dataset Saved Successfully")

Risk Dataset Saved Successfully


In [22]:
df.shape

(6362620, 33)

In [23]:
df["fraud_severity"].value_counts()

fraud_severity
None        6354427
Extreme        6630
Severe         1248
Moderate        282
Minor            33
Name: count, dtype: int64